In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from utils import classify_sentiment, normalize_text

df = pd.read_csv('../data/ai_apps_reviews_100000.csv')

if 'Sentiment' not in df.columns:
    df['Sentiment'] = df['score'].apply(classify_sentiment)

spacy_df = df.dropna(subset=['appName', 'content', 'score', 'Sentiment']).copy()
spacy_df['content'] = spacy_df['content'].apply(normalize_text)
spacy_df = spacy_df[spacy_df['content'] != ''].reset_index(drop=True)

print('Rows:', len(spacy_df))
print('Sentiment counts:', spacy_df['Sentiment'].value_counts().to_dict())

Rows: 233512
Sentiment counts: {'Positive': 194641, 'Negative': 28565, 'Neutral': 10306}


In [2]:
# Lemmatization + POS tagging (spaCy)

import spacy

nlp = spacy.load('en_core_web_sm')

sample_n = min(2000, len(spacy_df))
spacy_sample = spacy_df.sample(n=sample_n, random_state=42).copy()

def spacy_lemma_and_pos(text: str):
    doc = nlp(text)
    lemmas = [t.lemma_.lower() for t in doc if not t.is_space]
    pos = [t.pos_ for t in doc if not t.is_space]
    return ' '.join(lemmas), pos

lemmas = []
pos_tags = []
for txt in spacy_sample['content'].tolist():
    l, p = spacy_lemma_and_pos(txt)
    lemmas.append(l)
    pos_tags.append(p)

spacy_sample['lemma_content'] = lemmas
spacy_sample['pos_tags'] = pos_tags

def pos_dist(tag_lists):
    from collections import Counter
    c = Counter()
    for tags in tag_lists:
        c.update(tags)
    total = sum(c.values()) or 1
    return {k: v / total for k, v in c.most_common(10)}

print('Top POS tags by sentiment (sample):')
for sent in ['Negative', 'Neutral', 'Positive']:
    sub = spacy_sample[spacy_sample['Sentiment'] == sent]
    print(sent, 'n=', len(sub), pos_dist(sub['pos_tags'].tolist()))

Top POS tags by sentiment (sample):
Negative n= 258 {'NOUN': 0.1656764168190128, 'VERB': 0.1387111517367459, 'PRON': 0.10580438756855576, 'PUNCT': 0.0850091407678245, 'ADJ': 0.08249542961608775, 'AUX': 0.06992687385740402, 'ADV': 0.06444241316270567, 'ADP': 0.06261425959780621, 'PROPN': 0.061243144424131625, 'DET': 0.054387568555758686}
Neutral n= 93 {'NOUN': 0.16483516483516483, 'VERB': 0.12454212454212454, 'ADJ': 0.10256410256410256, 'PRON': 0.09816849816849817, 'PUNCT': 0.07985347985347985, 'AUX': 0.0652014652014652, 'DET': 0.06446886446886448, 'ADP': 0.063003663003663, 'ADV': 0.06153846153846154, 'PROPN': 0.05347985347985348}
Positive n= 1649 {'NOUN': 0.16203481103952302, 'ADJ': 0.15789979805750554, 'PRON': 0.10289450908741225, 'VERB': 0.09808635445715934, 'PROPN': 0.0834695643811905, 'ADV': 0.08221944417732474, 'PUNCT': 0.07808443119530724, 'ADP': 0.05308202711799211, 'AUX': 0.05250504856236177, 'DET': 0.04558130589479758}
